# Projet Kayak — Partie 3 : Scraping des hôtels Booking

Pour les **5 villes du top météo**, on scrape les **20 premiers hôtels** sur Booking.com avec **Scrapy**.

**Champs récupérés par hôtel :**
- nom, URL de la fiche, note Booking (sur 10), description, coordonnées GPS

**Architecture :**
- Le spider est dans `booking_scraper/` (projet Scrapy autonome).
- Ce notebook se contente de **lancer le spider** et de **recharger les résultats**.

## Contexte : diagnostic préalable du comportement anti-bot de Booking

Avant d'écrire ce spider en mode Playwright, un diagnostic a été fait pour comprendre pourquoi une approche Scrapy classique ne fonctionnait pas sur Booking.com.

L'outil utilisé est `scrapy shell`, qui ouvre un environnement interactif où on peut inspecter la réponse HTTP réelle renvoyée par un site — utile pour comprendre ce que Scrapy "voit" avant même d'écrire une ligne de spider.

**Commande lancée dans le terminal**, depuis le dossier `booking_scraper/` :

```bash
scrapy shell "https://www.booking.com/searchresults.fr.html?ss=Marseille"
```

**Inspection dans le shell interactif :**

>>> print(response.status)
202

>>> "challenge" in response.text

>>> "awsWafCookieDomainList" in response.text

>>> print(response.text[:500])

<!DOCTYPE html>
<html lang="en">
<head>
    <title></title>
    <script type="text/javascript" 
        src="https://www.booking.com/__challenge_h78IRKX3.../challenge.js">
    </script>
    <script>
        window.awsWafCookieDomainList = ['booking.com'];
    </script>
</head>
<body>
    <div id="challenge-container"></div>
</body>
</html>


**Interprétation :**

- Code HTTP **202** (Accepted) au lieu du 200 attendu → réponse dégradée
- HTML minimaliste (~500 caractères) au lieu de la page de résultats complète
- Présence d'un `<div id="challenge-container">` et d'un script `challenge.js` chargé depuis AWS WAF
- La variable JS `awsWafCookieDomainList` confirme qu'il s'agit d'un challenge d'**AWS WAF** (Web Application Firewall)

**Diagnostic :** Booking utilise un challenge JavaScript de fingerprinting (canvas, performances, comportement de la souris) pour valider que la session vient d'un vrai navigateur. Le challenge n'accorde un cookie de session valide qu'une fois résolu par exécution du JS. Or Scrapy pur n'exécute pas de JavaScript.

**Conclusion :** migration nécessaire vers `scrapy-playwright`, qui pilote un vrai navigateur Chromium en arrière-plan. Le challenge se résout transparentement et les requêtes suivantes reçoivent bien un HTTP 200 avec le HTML complet des résultats de recherche.


## 1. Vérification du top 5

Le spider va lire automatiquement `data/top5_cities.csv` pour savoir quelles villes scraper. On vérifie d'abord que ce fichier existe bien.

In [1]:
import pandas as pd

top5 = pd.read_csv("data/top5_cities.csv")
print("Villes qui seront scrapées :")
for city in top5["city"]:
    print(f"  - {city}")

Villes qui seront scrapées :
  - Marseille
  - Aix en Provence
  - Avignon
  - Nimes
  - Aigues Mortes


## 2. Lancement du spider Scrapy

On lance le spider via une commande shell. Scrapy va :
1. Lire les villes depuis `data/top5_cities.csv`
2. Pour chaque ville : interroger la page de recherche Booking, en extraire les 20 premiers hôtels
3. Pour chaque hôtel : visiter sa fiche pour récupérer description et GPS
4. Exporter le tout dans `data/hotels.json`

**Durée estimée :** ~3 à 5 minutes (à cause du `DOWNLOAD_DELAY=2` configuré dans `settings.py` pour ne pas se faire bloquer).

In [2]:
import os

# On supprime l'ancien fichier de sortie si présent : Scrapy fait de l'append par défaut,
# ce qui produirait un JSON invalide en cas de relance.
if os.path.exists("data/hotels.json"):
    os.remove("data/hotels.json")

# Lancement du spider depuis le dossier booking_scraper/
# -o data/hotels.json : chemin de sortie (relatif à booking_scraper/)
!cd booking_scraper && scrapy crawl booking -o ../data/hotels.json

2026-06-28 19:13:26 [scrapy.utils.log] INFO: Scrapy 2.16.0 started (bot: booking_scraper)
2026-06-28 19:13:26 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.11.9',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 '
           '64 bit (AMD64)]',
 'pyOpenSSL': '26.2.0 (OpenSSL 4.0.0 14 Apr 2026)',
 'cryptography': '48.0.0',
 'Platform': 'Windows-11-10.0.26200-SP0'}
2026-06-28 19:13:26 [scrapy.addons] INFO: Enabled addons:
[]
2026-06-28 19:13:26 [scrapy.extensions.telnet] INFO: Telnet Password: 90fa2412b019b8df
2026-06-28 19:13:26 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-06-28 19:13:26 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME'

## 3. Chargement et inspection des résultats

In [4]:
df_hotels = pd.read_json("data/hotels.jsonl", lines=True)
print(f"Nombre total d'hôtels scrapés : {len(df_hotels)}")
print(f"\nNombre d'hôtels par ville :")
print(df_hotels["city"].value_counts())

Nombre total d'hôtels scrapés : 100

Nombre d'hôtels par ville :
city
Strasbourg         20
Aix en Provence    20
Marseille          20
Lyon               20
Besancon           20
Name: count, dtype: int64


In [5]:
# Aperçu des données
df_hotels.head()

,city,name,url,score,lat,lon
0,Strasbourg,Budget Hôtel Weber - Strasbourg Centre Gare,https://www.booking.com/hotel/fr/weber.fr.html...,7.2,48.581243,7.732145
1,Strasbourg,Séjours & Affaires Strasbourg Kleber,https://www.booking.com/hotel/fr/residhome-str...,7.8,48.583197,7.742852
2,Strasbourg,City Résidence Strasbourg Centre,https://www.booking.com/hotel/fr/appart-victor...,7.8,48.589062,7.738799
3,Strasbourg,ibis Styles Strasbourg Avenue du Rhin,https://www.booking.com/hotel/fr/ibis-strasbou...,8.3,48.569366,7.777569
4,Strasbourg,Residence Inn by Marriott Strasbourg,https://www.booking.com/hotel/fr/residence-inn...,8.8,48.598995,7.762173


In [6]:
# Vérification de la qualité : combien de valeurs manquantes par colonne ?
df_hotels.isna().sum()

city      0
name      0
url       0
score    10
lat       0
lon       0
dtype: int64

## 4. Nettoyage léger

On supprime les hôtels sans coordonnées GPS (impossibles à afficher sur la carte) et on trie par note décroissante par ville.

In [10]:
# On garde uniquement les hôtels avec lat/lon valides
# On exclut les hôtels sans coordonnées GPS ET sans note Booking
df_hotels_clean = df_hotels.dropna(subset=["lat", "lon", "score"]).copy()
print(f"Hôtels conservés : {len(df_hotels_clean)} / {len(df_hotels)}")

# Tri par ville puis par note décroissante
df_hotels_clean = df_hotels_clean.sort_values(
    ["city", "score"], ascending=[True, False]
).reset_index(drop=True)

print(f"Hôtels conservés : {len(df_hotels_clean)} / {len(df_hotels)}")
df_hotels_clean.head(10)

Hôtels conservés : 90 / 100
Hôtels conservés : 90 / 100


,city,name,url,score,lat,lon
0,Aix en Provence,Studio terrasse,https://www.booking.com/hotel/fr/studio-terras...,9.9,43.518807,5.458397
1,Aix en Provence,Hôtel Boutique Cézanne centre Aix-en-Provence,https://www.booking.com/hotel/fr/cezanne-aix-e...,8.7,43.523562,5.445937
2,Aix en Provence,Domaine Gao,https://www.booking.com/hotel/fr/domaine-and-c...,8.7,43.496401,5.394397
3,Aix en Provence,Hotel Cardinal,https://www.booking.com/hotel/fr/cardinal.fr.h...,8.6,43.525599,5.451721
4,Aix en Provence,Renaissance Aix-en-Provence Hotel,https://www.booking.com/hotel/fr/renaissance-a...,8.6,43.526331,5.437964
5,Aix en Provence,Hôtel Birdy by Happyculture,https://www.booking.com/hotel/fr/royal-mirabea...,8.6,43.481430,5.365614
6,Aix en Provence,Domaine de Carraire,https://www.booking.com/hotel/fr/domaine-de-ca...,8.6,43.566063,5.384448
7,Aix en Provence,Villa avec piscine et jacuzzi,https://www.booking.com/hotel/fr/villa-avec-pi...,8.5,43.566364,5.442922
8,Aix en Provence,L'Escapade Aixoise hyper centre historique stu...,https://www.booking.com/hotel/fr/l-39-escapade...,8.5,43.531203,5.449408
9,Aix en Provence,Hôtel Le Mozart,https://www.booking.com/hotel/fr/le-mozart.fr....,8.4,43.521918,5.457872


In [11]:
# Sauvegarde de la version nettoyée
df_hotels_clean.to_csv("data/hotels.csv", index=False)
print("✅ Données sauvegardées dans data/hotels.csv")

✅ Données sauvegardées dans data/hotels.csv


## 5. Visualisation : carte des hôtels du top 5

Pour chaque ville du top 5 météo, on affiche les hôtels avec un marqueur coloré selon la note Booking.

In [12]:
import plotly.express as px

fig = px.scatter_map(
    df_hotels_clean,
    lat="lat",
    lon="lon",
    color="score",                        # couleur selon la note Booking
    size="score",                         # taille selon la note
    size_max=15,
    hover_name="name",
    hover_data={
        "city": True,
        "score": ":.1f",
        "lat": False,
        "lon": False,
    },
    color_continuous_scale="RdYlGn",      # rouge → jaune → vert : note basse → haute
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    height=600,
    title="Hôtels Booking dans les 5 meilleures destinations météo",
)

fig.update_layout(map_style="carto-positron")
fig.show()